# Phase 7 — Composite score & clustering

**Composite S:** When solar yield CV R² < 0, **P50_norm** uses `physics_hybrid_cf_p50` from `01b` (physics fallback) instead of XGB `hybrid_cf_p50`.

```
S = 0.35×resource + 0.15×seasonal + 0.30×P50 − 0.10×risk + 0.10×developability
```

| Output |
|--------|
| `cell_composite_score.parquet` |
| `cell_clusters.parquet` |
| `outputs/ny_composite_ranked.geojson` |
| `maps/composite_score_ny.html` |

**All logic runs in this notebook** (no external script imports).

**Requires:** `06`, `09`, `10`, `11`, `12`

In [1]:
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import plotly.graph_objects as go
import polars as pl
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = Path('../..').resolve()
GATEWAY_PATH = ROOT / 'data' / 'processed' / 'cells_passing_gateway.parquet'
RESOURCE_PATH = ROOT / 'data' / 'processed' / 'cell_resource_scores.parquet'
SEASONAL_PATH = ROOT / 'data' / 'processed' / 'hybrid_suitability.parquet'
P50_PATH = ROOT / 'data' / 'processed' / 'cell_p50_yield.parquet'
PHYSICS_PATH = ROOT / 'data' / 'processed' / 'cell_physics_yield.parquet'
YIELD_SOLAR_CV_PATH = ROOT / 'data' / 'processed' / 'yield_solar_cv_report.json'
RISK_PATH = ROOT / 'data' / 'processed' / 'cell_risk_spread.parquet'
OUT_SCORE = ROOT / 'data' / 'processed' / 'cell_composite_score.parquet'
OUT_CLUSTER = ROOT / 'data' / 'processed' / 'cell_clusters.parquet'
OUT_GEOJSON = ROOT / 'outputs' / 'ny_composite_ranked.geojson'
REPORT_PATH = ROOT / 'data' / 'processed' / 'composite_score_report.json'
MAP_PATH = ROOT / 'maps' / 'composite_score_ny.html'
NY_BOUNDARY_PATH = ROOT / 'data' / 'ref' / 'ny_state_boundary.geojson'

W_RESOURCE, W_SEASONAL, W_YIELD, W_RISK, W_LAND = 0.35, 0.15, 0.30, 0.10, 0.10
N_CLUSTERS, RANDOM_STATE = 4, 42
URBAN_FRACS = ['frac_class_21', 'frac_class_22', 'frac_class_23', 'frac_class_24']

def pct_100(col: str) -> pl.Expr:
    return (pl.col(col).rank() / pl.col(col).count()) * 100.0

print('ROOT:', ROOT)

ROOT: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York


In [2]:
gw_raw = pl.read_parquet(GATEWAY_PATH).filter(pl.col('in_ny_state'))
gw_cols = ['latitude', 'longitude', 'frac_class_11', 'dist_to_transmission_km'] + URBAN_FRACS
gw = gw_raw.select([c for c in gw_cols if c in gw_raw.columns])
gw = gw.with_columns(pl.sum_horizontal([pl.col(c).fill_null(0) for c in URBAN_FRACS]).alias('urban_frac'))

df = gw
for path, cols in [
    (RESOURCE_PATH, ['hybrid_screening_score']),
    (SEASONAL_PATH, ['hybrid_seasonal_score', 'jackpot_zone']),
    (P50_PATH, ['hybrid_cf_p50']),
    (RISK_PATH, ['hybrid_risk_spread', 'solar_stress_ratio_std', 'wind_stress_ratio_std']),
]:
    if path.exists():
        sub = pl.read_parquet(path)
        pick = ['latitude', 'longitude'] + [c for c in cols if c in sub.columns]
        df = df.join(sub.select(pick), on=['latitude', 'longitude'], how='inner')

# physics fallback: use 01b P50 when XGB solar yield CV is negative
P50_SOURCE = 'xgb'
if YIELD_SOLAR_CV_PATH.exists():
    solar_cv_r2 = json.loads(YIELD_SOLAR_CV_PATH.read_text()).get('cv_oof', {}).get('r2', 0.0)
else:
    solar_cv_r2 = 0.0

if solar_cv_r2 < 0 and PHYSICS_PATH.exists():
    phys = pl.read_parquet(PHYSICS_PATH).select([
        'latitude', 'longitude', 'physics_hybrid_cf_p50',
    ])
    df = df.join(phys, on=['latitude', 'longitude'], how='left')
    df = df.with_columns(
        pl.when(pl.col('physics_hybrid_cf_p50').is_not_null())
        .then(pl.col('physics_hybrid_cf_p50'))
        .otherwise(pl.col('hybrid_cf_p50'))
        .alias('P50_for_composite')
    )
    P50_SOURCE = 'physics_blend_negative_xgb_cv'
else:
    df = df.with_columns(pl.col('hybrid_cf_p50').alias('P50_for_composite'))

print('Merged cells:', df.height)
print(f'P50 source: {P50_SOURCE} (solar yield CV R²={solar_cv_r2:.3f})')
df.head(3)

Merged cells: 1089
P50 source: physics_blend_negative_xgb_cv (solar yield CV R²=-0.317)


latitude,longitude,frac_class_11,dist_to_transmission_km,frac_class_21,frac_class_22,frac_class_23,frac_class_24,urban_frac,hybrid_screening_score,hybrid_seasonal_score,jackpot_zone,hybrid_cf_p50,hybrid_risk_spread,solar_stress_ratio_std,wind_stress_ratio_std,physics_hybrid_cf_p50,P50_for_composite
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f32,f64,f64,f64,f64,f64
42.39,-79.26,0.006986,1.674329,0.056626,0.02388,0.003291,0.000357,0.084154,0.923783,0.891848,true,0.027,0.005193,0.018541,0.087253,0.119656,0.119656
41.94,-73.77,0.006242,2.422054,0.077935,0.017702,0.001938,0.000141,0.097716,0.535813,0.139385,false,0.04286,0.005537,0.023808,0.063283,0.094052,0.094052
43.11,-73.86,0.001148,1.393111,0.086892,0.071105,0.026391,0.007964,0.192353,0.405877,0.021482,false,0.023348,0.002962,0.02257,0.074568,0.092387,0.092387


In [3]:
df = df.with_columns([
    (1.0 - pl.col('frac_class_11').fill_null(0) - pl.col('urban_frac').fill_null(0)).clip(0, 1).alias('developability_raw'),
])
df = df.with_columns([
    pct_100('developability_raw').alias('D_norm'),
    pct_100('dist_to_transmission_km').alias('transmission_pct'),
])
df = df.with_columns([
    (100.0 - pl.col('transmission_pct')).alias('transmission_score'),
]).with_columns([
    ((pl.col('D_norm') + pl.col('transmission_score')) / 2).alias('D_norm'),
    pct_100('hybrid_screening_score').alias('resource_norm'),
    pct_100('hybrid_seasonal_score').alias('seasonal_norm'),
    pct_100('hybrid_cf_p50').alias('P50_norm_xgb'),
    pct_100('P50_for_composite').alias('P50_norm'),
    pct_100('hybrid_risk_spread').alias('R_norm'),
])

def composite_from_weights(w_resource, w_seasonal, w_yield, w_risk, w_land) -> pl.Expr:
    return (
        w_resource * pl.col('resource_norm')
        + w_seasonal * pl.col('seasonal_norm')
        + w_yield * pl.col('P50_norm')
        - w_risk * pl.col('R_norm')
        + w_land * pl.col('D_norm')
    )

PERSONAS = {
    'balanced': (0.35, 0.15, 0.30, 0.10, 0.10),
    'resource_first': (0.55, 0.15, 0.15, 0.05, 0.10),
    'conservative': (0.25, 0.15, 0.20, 0.25, 0.15),
    'yield_focus': (0.20, 0.10, 0.45, 0.10, 0.15),
    'risk_plus_10pct': (0.35, 0.15, 0.25, 0.15, 0.10),
    'resource_plus_10pct': (0.45, 0.15, 0.25, 0.10, 0.05),
}

df = df.with_columns(
    composite_from_weights(W_RESOURCE, W_SEASONAL, W_YIELD, W_RISK, W_LAND).alias('composite_score')
)
for persona_name, (wr, ws, wy, wk, wl) in PERSONAS.items():
    col = f'composite_persona_{persona_name}'
    df = df.with_columns(composite_from_weights(wr, ws, wy, wk, wl).alias(col))
    df = df.with_columns(pl.col(col).rank(method='average').alias(f'rank_persona_{persona_name}'))

df = df.with_columns([
    (pl.col('rank_persona_balanced') - pl.col('rank_persona_resource_first')).alias('rank_delta_resource_first'),
    (pl.col('rank_persona_balanced') - pl.col('rank_persona_conservative')).alias('rank_delta_conservative'),
])
df = df.with_columns(pct_100('composite_score').alias('composite_rank_pct'))

In [4]:
feat = df.select(['D_norm', 'P50_norm', 'R_norm']).to_numpy()
X = StandardScaler().fit_transform(feat)
labels = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit_predict(X)
sil = float(silhouette_score(X, labels))

cluster_names = {0: 'A_prime_balanced', 1: 'B_high_yield_risk', 2: 'C_moderate_safe', 3: 'D_lower_tier'}
df = df.with_columns(pl.Series('cluster_id', labels))
df = df.with_columns(pl.col('cluster_id').replace(cluster_names, default='unknown').alias('cluster_label'))

df.write_parquet(OUT_SCORE)
df.select(['latitude', 'longitude', 'cluster_id', 'cluster_label', 'composite_score']).write_parquet(OUT_CLUSTER)

OUT_GEOJSON.parent.mkdir(parents=True, exist_ok=True)
gpd.GeoDataFrame(
    df.to_pandas(),
    geometry=gpd.points_from_xy(df['longitude'].to_numpy(), df['latitude'].to_numpy()),
    crs='EPSG:4326',
).to_file(OUT_GEOJSON, driver='GeoJSON')

print(f'Wrote {OUT_SCORE}')
print(f'Wrote {OUT_GEOJSON}')
print(f'Silhouette: {sil:.3f}')

Wrote /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/cell_composite_score.parquet
Wrote /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/outputs/ny_composite_ranked.geojson
Silhouette: 0.284


In [5]:
centroids = []
for cid in range(N_CLUSTERS):
    sub = df.filter(pl.col('cluster_id') == cid)
    centroids.append({
        'cluster_id': cid, 'label': cluster_names[cid], 'n_cells': sub.height,
        'mean_composite': float(sub['composite_score'].mean()),
    })

report = {
    'n_cells': df.height,
    'roadmap2_p50_source': P50_SOURCE,
    'solar_yield_cv_r2': float(solar_cv_r2),
    'weights': {'resource': W_RESOURCE, 'seasonal': W_SEASONAL, 'yield': W_YIELD, 'risk': W_RISK, 'land': W_LAND},
    'silhouette': sil, 'cluster_centroids': centroids,
    'top_10': df.sort('composite_score', descending=True).head(10).to_dicts(),
}
REPORT_PATH.write_text(json.dumps(report, indent=2, default=float), encoding='utf-8')

for c in centroids:
    print(f"  {c['label']}: n={c['n_cells']} mean S={c['mean_composite']:.1f}")
df.sort('composite_score', descending=True).head(10)

  A_prime_balanced: n=238 mean S=28.0
  B_high_yield_risk: n=276 mean S=59.6
  C_moderate_safe: n=280 mean S=18.4
  D_lower_tier: n=295 mean S=52.0


latitude,longitude,frac_class_11,dist_to_transmission_km,frac_class_21,frac_class_22,frac_class_23,frac_class_24,urban_frac,hybrid_screening_score,hybrid_seasonal_score,jackpot_zone,hybrid_cf_p50,hybrid_risk_spread,solar_stress_ratio_std,wind_stress_ratio_std,physics_hybrid_cf_p50,P50_for_composite,developability_raw,D_norm,transmission_pct,transmission_score,resource_norm,seasonal_norm,P50_norm_xgb,P50_norm,R_norm,composite_score,composite_rank_pct,cluster_id,cluster_label
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,str
42.57,-79.08,0.142839,0.049348,0.060929,0.0557,0.013199,0.001558,0.131385,0.995409,0.98901,true,0.024806,0.003324,0.017791,0.081804,0.157038,0.157038,0.725776,51.698806,1.469238,98.530762,99.632691,99.632691,26.721763,99.540863,42.424242,80.606061,100.0,1,"""B_high_yield_risk"""
42.21,-79.71,0.00171,1.483469,0.057315,0.02151,0.005376,0.00106,0.085262,0.990358,0.988088,true,0.020241,0.004067,0.018985,0.096537,0.134034,0.134034,0.913028,67.309458,26.81359,73.18641,99.449036,99.540863,6.244261,98.898072,60.697888,80.068871,99.908173,3,"""D_lower_tier"""
42.3,-79.53,0.020883,0.435231,0.076798,0.043816,0.009387,0.002272,0.132273,0.995868,0.993579,true,0.021544,0.003802,0.018582,0.087089,0.153407,0.153407,0.846844,55.73921,9.641873,90.358127,99.770432,99.770432,10.927456,99.449036,55.004591,79.793388,99.816345,1,"""B_high_yield_risk"""
43.47,-76.47,0.339591,1.240054,0.071763,0.065714,0.042332,0.016022,0.195831,0.891185,0.971582,true,0.020349,0.002983,0.013317,0.078658,0.16526,0.16526,0.464578,38.705234,22.865014,77.134986,97.796143,99.081726,6.97888,99.724518,31.680441,79.710744,99.724518,1,"""B_high_yield_risk"""
42.48,-79.26,0.143777,0.008468,0.07019,0.074166,0.03667,0.012549,0.193576,0.997704,0.995412,true,0.026785,0.004006,0.018185,0.084854,0.17459,0.17459,0.662647,51.285583,0.275482,99.724518,99.908173,99.908173,33.700643,99.908173,59.59596,79.0955,99.632691,1,"""B_high_yield_risk"""
42.48,-79.17,0.001856,3.404297,0.063325,0.035884,0.005862,0.000223,0.105293,0.988981,0.979887,true,0.021172,0.003589,0.01819,0.091439,0.139652,0.139652,0.892851,44.674013,52.984389,47.015611,99.357208,99.357208,9.366391,99.265381,49.035813,79.022039,99.540863,1,"""B_high_yield_risk"""
42.66,-78.18,0.002635,3.47439,0.04123,0.027838,0.003777,0.001013,0.073859,0.747475,0.816058,true,0.019972,0.002388,0.019049,0.07792,0.125271,0.125271,0.923506,59.22865,53.902663,46.097337,89.715335,95.684114,5.325987,97.428834,19.008264,79.003673,99.449036,1,"""B_high_yield_risk"""
43.83,-76.2,0.242015,10.282667,0.034843,0.027002,0.003555,0.000761,0.066161,0.962351,0.96522,true,0.026107,0.00125,0.009563,0.041094,0.168196,0.168196,0.691824,4.820937,94.031221,5.968779,99.035813,98.898072,31.588613,99.816345,9.366391,78.987603,99.357208,1,"""B_high_yield_risk"""
43.38,-76.47,0.023839,0.287586,0.093804,0.045892,0.010189,0.002711,0.152596,0.857208,0.953711,true,0.02508,0.003418,0.014155,0.085595,0.131223,0.131223,0.823565,54.637282,6.887052,93.112948,96.235078,98.438935,27.731864,98.347107,45.087236,78.907254,99.265381,1,"""B_high_yield_risk"""


In [6]:
m = df.to_pandas()
ny = gpd.read_file(NY_BOUNDARY_PATH)
outline = ny.boundary.iloc[0]

fig = go.Figure()
fig.add_trace(go.Scattermap(
    lat=m['latitude'], lon=m['longitude'], mode='markers',
    marker=dict(size=8, color=m['composite_score'], colorscale='Turbo', showscale=True, opacity=0.75),
))
fig.add_trace(go.Scattermap(
    lat=list(outline.xy[1]), lon=list(outline.xy[0]),
    mode='lines', line=dict(width=2, color='#333'), hoverinfo='skip', showlegend=False,
))
fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=42.9, lon=-75.8), zoom=5.5),
    title='Composite investment score',
    height=800,
)
MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(str(MAP_PATH), include_plotlyjs='cdn')
print(f'Map -> {MAP_PATH}')

Map -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/maps/composite_score_ny.html
